In [49]:
import pandas as pd

# Učitavanje sheetova-a i podaci o fajlu

In [50]:
df = pd.read_excel("../telekom_korisnici.xlsx",sheet_name=["Podaci","Rečnik_podataka"])
data_dict = df["Rečnik_podataka"]
data = df["Podaci"]

In [51]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1232 entries, 0 to 1231
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_korisnika        1232 non-null   str    
 1   region              1208 non-null   str    
 2   pol                 1232 non-null   str    
 3   godine              1184 non-null   float64
 4   tip_paketa          1232 non-null   str    
 5   staz_meseci         1232 non-null   int64  
 6   mesecni_racun_rsd   1196 non-null   object 
 7   potrosnja_gb        1196 non-null   object 
 8   minuti_poziva       1232 non-null   int64  
 9   broj_sms            1232 non-null   int64  
 10  broj_reklamacija    1232 non-null   int64  
 11  kanal_prodaje       1220 non-null   str    
 12  datum_aktivacije    1232 non-null   str    
 13  ocena_zadovoljstva  1136 non-null   float64
 14  churn               1232 non-null   str    
dtypes: float64(2), int64(4), object(2), str(7)
memory usage: 144.5+ KB

In [52]:
display(data_dict)

,kolona,tip,opis,dozvoljene vrednosti
0,id_korisnika,tekst,Jedinstveni identifikator korisnika (K00001…),jedinstven
1,region,kategorija,Region u kojem je korisnik registrovan,Beograd; Vojvodina; Šumadija i Zapadna Srbija;...
2,pol,kategorija,Pol korisnika,M; Ž
3,godine,ceo broj,Starost korisnika u godinama,18–99
4,tip_paketa,kategorija,Tip tarifnog paketa,Prepaid; Postpaid; Biznis
5,staz_meseci,ceo broj,Broj meseci od aktivacije do 01.09.2026.,≥ 1
6,mesecni_racun_rsd,decimalni broj,Prosečan mesečni račun u dinarima (poslednjih ...,> 0
7,potrosnja_gb,decimalni broj,Prosečna mesečna potrošnja mobilnih podataka u GB,0–150
8,minuti_poziva,ceo broj,Prosečan broj minuta odlaznih poziva mesečno,≥ 0
9,broj_sms,ceo broj,Prosečan broj poslatih SMS poruka mesečno,≥ 0


Dimenzije su: `1232x15` (1232 reda, 15 kolona)

Tipovi kako ih vidi `pandas` su prikazani u prvom box-u ispod čega je prikazano željeno stanje.

Falinke (po meni):

- `godine` su zapravo `int64`, a ne `float64`
- `mesecni_racun_rsd` je `float64`, a ne `object`
- `potrosnja_gb` je `float64`
- `minuti_poziva` su `int64`
- `datum_aktivacije` je `datetime64`
- `ocena_zadovoljstva` je `int64`

# Profil kvaliteta podataka

In [53]:
#nedostajuće vrednosti, duplikati,
#jedinstvene vrednosti kategorijskih kolona i ne-numerički unosi u numeričkim kolonama
# POMOC ZA MENE

missing_values = data.isna().sum()

missing_values=missing_values[missing_values>0]

total_duplicates = data.duplicated().sum()

id_duplicates = data["id_korisnika"].duplicated().sum()

categories=["region", "pol", "tip_paketa", "kanal_prodaje", "churn"]

for cat in categories:
    print("{} {} {}".format(cat,data[cat].nunique(),data[cat].value_counts(dropna=False).to_dict()))
print("\n")
for field in ["mesecni_racun_rsd","potrosnja_gb"]: #znamo da od ovih realno i ima vajde (videlo se ranije)
    non_num = pd.to_numeric(data[field],errors="coerce").isna() & data[field].notna()
    print(field,non_num.sum(),data.loc[non_num,field].head().to_list())

rows=[]

for col in data.columns:
    # imamo broj nanova gore u missing_values
    nan_number = data[col].isna().sum()
    nan_prop = nan_number/len(data)
    unique_number = data[col].nunique()

    non_num = (pd.to_numeric(data[col],errors="coerce").isna() &
               data[col].notna())

    rows.append({
        "kolona":col,
        "tip":str(data[col].dtype),
        "broj_nan":nan_number,
        "udeo_nan":nan_prop,
        "broj_jedinstvenih":unique_number,
        "broj_nenum":non_num.sum(),
        "primer_problema":data.loc[non_num,col].head(5).to_list()
    })


quality_prof = pd.DataFrame(rows)

display(quality_prof)
quality_prof.to_csv("profil_podataka.csv",index=False)

display(data.describe().loc[['min','max']])

region 17 {'Beograd': 390, 'Vojvodina': 305, 'Južna i Istočna Srbija': 226, 'Šumadija i Zapadna Srbija': 225, nan: 24, 'VOJVODINA': 9, 'šumadija i zapadna srbija': 8, 'Juzna i Istocna Srbija': 8, 'Sumadija i Zapadna Srbija': 8, 'BEOGRAD': 6, 'južna i istočna srbija': 5, 'vojvodina': 5, 'Beograd ': 4, 'Vojvodina ': 3, 'beograd': 2, 'BG': 2, 'JUŽNA I ISTOČNA SRBIJA': 1, 'ŠUMADIJA I ZAPADNA SRBIJA': 1}
pol 10 {'M': 621, 'Ž': 581, 'm': 8, 'Z': 4, 'muski': 4, 'M ': 4, 'Muški': 4, 'ž': 3, 'zenski': 2, 'Ženski': 1}
tip_paketa 12 {'Postpaid': 576, 'Prepaid': 454, 'Biznis': 172, 'Post-paid': 7, 'POSTPAID': 6, 'Pre-paid': 4, 'BIZNIS': 4, 'postpaid': 4, 'biznis': 2, 'PREPAID': 1, 'Business': 1, 'prepaid': 1}
kanal_prodaje 12 {'Poslovnica': 505, 'Online': 376, 'Call centar': 167, 'Partner': 148, nan: 12, ' Poslovnica ': 7, ' Online ': 5, 'poslovnica': 4, 'online': 3, 'call centar': 2, 'partner': 1, ' Call centar ': 1, ' Partner ': 1}
churn 11 {'Ne': 927, 'Da': 265, '0': 13, 'NE': 8, 'Ne ': 6, '1':

,kolona,tip,broj_nan,udeo_nan,broj_jedinstvenih,broj_nenum,primer_problema
0,id_korisnika,str,0,0.000000,1200,1232,"[K00900, K00523, K00536, K00571, K00677]"
1,region,str,24,0.019481,17,1208,"[Šumadija i Zapadna Srbija, Južna i Istočna Sr..."
2,pol,str,0,0.000000,10,1232,"[M, Ž, Ž, M, M]"
3,godine,float64,48,0.038961,67,0,[]
4,tip_paketa,str,0,0.000000,12,1232,"[Prepaid, Postpaid, Postpaid, Postpaid, Prepaid]"
5,staz_meseci,int64,0,0.000000,160,0,[]
6,mesecni_racun_rsd,object,36,0.029221,1027,34,"[2235 RSD, 3.274,00, 1.861 din, 2.549,00, 1250..."
7,potrosnja_gb,object,36,0.029221,248,18,"[4,5, 1,8, 6,4, 10,2, 7,5]"
8,minuti_poziva,int64,0,0.000000,449,0,[]
9,broj_sms,int64,0,0.000000,37,0,[]


,godine,staz_meseci,minuti_poziva,broj_sms,broj_reklamacija,ocena_zadovoljstva
min,-26.0,-9.0,-139.0,20.0,0.0,1.0
max,150.0,180.0,7500.0,59.0,7.0,5.0


Očito je da se opsezi ne slažu i to u `minuti_poziva`, `staz_meseci` i `godine`.

### Problem sa ID-jevima

Tražena razlika i nije baš da je stina: ID `K00013` ima dva skoro pa identična upisa - razlika je u tome što je `region` ukucan velikim odnosno malim slovima.

**PITANJE ZA MENTORA**:
- Kod ostalih razlike **ne vidim** koje ih čine (ne na oko)

- Primer problema: Mogu da stavim sve i svašta. Na kraju krajeva mogu i ručno da odaberem. Šta savetuješ da bi se algoritamski odabrao prihvatljiv primer problema ? `df.head()` je samo zakrpa koja omogućava da se `.csv` uopšte i formira.

In [54]:
print("Pronalazak duplikata po ID-ju:")
id_duplicates_alt = data[data.id_korisnika.duplicated(keep=False)].sort_values("id_korisnika")
display(id_duplicates_alt)

Pronalazak duplikata po ID-ju:


,id_korisnika,region,pol,godine,tip_paketa,staz_meseci,mesecni_racun_rsd,potrosnja_gb,minuti_poziva,broj_sms,broj_reklamacija,kanal_prodaje,datum_aktivacije,ocena_zadovoljstva,churn
712,K00013,ŠUMADIJA I ZAPADNA SRBIJA,Ž,40.0,Prepaid,49,1682,3,317,42,2,partner,2022-07-30,2.0,Da
881,K00013,Šumadija i Zapadna Srbija,Ž,40.0,Prepaid,49,1682,3,317,42,2,Partner,2022-07-30,2.0,Da
42,K00049,Šumadija i Zapadna Srbija,M,38.0,Postpaid,27,1477,5.2,337,32,1,Poslovnica,2024-05-29,4.0,Ne
135,K00049,Šumadija i Zapadna Srbija,M,38.0,Postpaid,27,1477,5.2,337,32,1,Poslovnica,2024-05-29,4.0,Ne
32,K00085,Beograd,Ž,18.0,Postpaid,85,2627,4.7,9,36,0,Call centar,2019-07-28,5.0,Ne
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,K01088,Beograd,M,57.0,Postpaid,20,2605,1.6,174,32,1,Online,2024-12-23,4.0,Ne
962,K01101,Vojvodina,M,51.0,Postpaid,6,2314,3.3,162,38,2,Poslovnica,2026-02-27,2.0,Da
475,K01101,Vojvodina,M,51.0,Postpaid,6,2314,3.3,162,38,2,Poslovnica,2026-02-27,2.0,Da
79,K01119,Vojvodina,M,18.0,Postpaid,62,2811,12.2,373,41,0,Partner,2021-06-30,2.0,Da


## Problemi sa datumima

Nakon ručnog trklejanja uviđa se da figurišu više-manje formati koji su sadržani u rečniku `formats`.

In [55]:
#col="datum_aktivacije"
format_ok = data["datum_aktivacije"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)

incorrect_format= data.loc[~format_ok,"datum_aktivacije"]

formats = {
    "ISO": r"^\d{4}-\d{2}-\d{2}$",
    "DD.MM.YYYY.": r"^\d{2}\.\d{2}\.\d{4}\.$",
    "MM/DD/YYYY": r"^\d{2}/\d{2}/\d{4}$"
}

date_categories=dict()

for name, pattern in formats.items():
    cond = data["datum_aktivacije"].str.match(pattern, na=False)
    date_categories[name]=cond.sum()

print(date_categories)


{'ISO': np.int64(1062), 'DD.MM.YYYY.': np.int64(110), 'MM/DD/YYYY': np.int64(60)}


# Mere centralne - ručno

In [56]:
import numpy as np

def modus(x:pd.Series):
    x_sort = np.sort(x)

    values,counts = np.unique(x_sort,return_counts=True) # mapa (vrednost->pozicija)

    position = np.where(counts==np.max(counts)) # nasao poziciju preko np

    position = np.argmax(position) #nasao max preko np

    return values[position]

def median(x:pd.Series):
    median=None
    x_sort = np.sort(x)
    n = x.size
    if n % 2 == 0:
        return ((x_sort[n//2 - 1] + x_sort[n//2]) / 2)
    else:
        return x_sort[n//2]

def central_tend_generator(x:pd.Series)->dict:
    x1 = x.copy()
    x = pd.to_numeric(x).dropna().to_numpy()

    return {
        "sredina":{"numpy":np.sum(x)/(x.size-1),"pandas":x1.describe().loc['mean']},
        "medijana":{"numpy":median(x),"pandas":x1.median()},
        "modus":{"numpy":modus(x),"pandas":x1.mode()}
    }

centralna=central_tend_generator(data["godine"])
display(centralna)

{'sredina': {'numpy': np.float64(42.273034657650044),
  'pandas': np.float64(42.23733108108108)},
 'medijana': {'numpy': np.float64(42.5), 'pandas': np.float64(42.5)},
 'modus': {'numpy': np.float64(-26.0),
  'pandas': 0    18.0
  Name: godine, dtype: float64}}

Mada se moj način rada totalno slaže sa predloženim rešenjem, brine me i dalje činjenica što su `godine` učitane kao negativni brojevi pa ne dobijam smislene rezultate.

- Prosek dobijen kroz `numpy` nešto veći u odnosu na `pandas` jer ovaj drugi ignoriše `NaN` pa zato izbacuje i manji prosek.
- Medijana je skroz ista - na prezentaciji smo utvrdili da $Me$ ima robusnost prema obimu uzorka
- Modus se razlikuje zato što radi sa najčešćom vrednošću, a izbacivanje `NaN` menja poredak u nizu i samim tim i njegovu vrednost.

### Otežani prosek

In [57]:
def weighted_mean():
    years_old = pd.to_numeric(data["godine"],errors="coerce")

    months = pd.to_numeric(data["staz_meseci"],errors="coerce")

    valid_rec = years_old.notna() & months.notna() #nemoguce ako je NaN neka od njih

    return {
        'weighted_mean': np.average(years_old[valid_rec],weights=months[valid_rec])
    }

results=weighted_mean()
display(results)


{'weighted_mean': np.float64(43.22881390261687)}

Neizbacivanje `NaN`-a daje `NaN` kao rezultat

In [58]:
years_old = pd.to_numeric(data["godine"],errors="coerce")
new_mean = np.nanmean(years_old)
print(new_mean,data["godine"].mean())

42.23733108108108 42.23733108108108


Vidimo da se u suštini ništa ne događa specijalno - obim uzorka je ostao isti. U prvom slučaju smo imali manje opservacija pa je i sredina morala da poraste, dok ovde to nije slučaj.

# Funkcija za sve mere

In [59]:
from scipy import stats

def mere(x:pd.Series)->dict:
    x=pd.to_numeric(x,errors="coerce").dropna().astype(float)

    positive = x[x>0]

    summary = stats.describe(x)

    q3=x.quantile(q=0.75)
    q1=x.quantile(q=0.25)

    return{
        "kolona":x.name,
        "n":len(x),
        "aritmeticka_sredina":x.mean(),
        "geometrijska_sredina":stats.gmean(x),
        "harmonijska_sredina":stats.hmean(x),
        "10_perc_trimovana":stats.trim_mean(x,proportiontocut=0.1),
        "raspon":x.describe().loc['max']-x.describe().loc['min'],
        "varijansa_uzorka":x.var(),
        "std_devijacija":x.std(),
        "IQR":q3-q1,
        "koeficijent_varijacije_perc":stats.variation(x),
        "asimetrija":stats.skew(x),
        "spljostenost":stats.kurtosis(x,bias=False),
        "q1":q1,
        "q3":q3,
        "p95":x.quantile(0.95),
        "mod":x.mode().iloc[0],
        "medijana":x.median()
    }

messages = mere(data["broj_sms"])
prigovori = mere(data["broj_reklamacija"])
print(messages["koeficijent_varijacije_perc"],prigovori["koeficijent_varijacije_perc"])

NUM = ["godine", "staz_meseci", "mesecni_racun_rsd", "potrosnja_gb",
       "minuti_poziva", "broj_sms", "broj_reklamacija", "ocena_zadovoljstva"]

values=[]
for col in NUM:
    values.append(mere(data[col]))

table= pd.DataFrame(values)
table.to_csv("mere_sirovo.csv",index=False)
#table.round(2).to_csv("mere_sirovo.csv")
display(table)

for col in NUM:
    record=mere(data[col])
    if abs(record["asimetrija"])>3:
        tmp=pd.to_numeric(data[col],errors="coerce")
        print("{} {}".format(col,tmp.nlargest(10).tolist()))

0.15991363675042733 1.3615166496564048


/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value encountered in log
  log_a = xp.log(a)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning: The harmonic mean is only defined if all elements are non-negative; otherwise, the result is NaN.
  res = hypotest_fun_out(*samples, **kwds)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value encountered in log
  log_a = xp.log(a)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning: The harmonic mean is only defined if all elements are non-negative; otherwise, the result is NaN.
  res = hypotest_fun_out(*samples, **kwds)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value enco

,kolona,n,aritmeticka_sredina,geometrijska_sredina,harmonijska_sredina,10_perc_trimovana,raspon,varijansa_uzorka,std_devijacija,IQR,koeficijent_varijacije_perc,asimetrija,spljostenost,q1,q3,p95,mod,medijana
0,godine,1184,42.237331,NaN,NaN,41.899789,176.0,2.227763e+02,14.925691,20.000,0.353227,0.856647,6.235530,32.0,52.000,65.00,18.0,42.5
1,staz_meseci,1232,41.548701,NaN,NaN,34.580122,189.0,1.560604e+03,39.504476,42.250,0.950413,1.609037,2.348816,14.0,56.250,132.00,3.0,29.0
2,mesecni_racun_rsd,1162,3006.548193,2313.349829,1876.919371,2516.592473,71740.0,1.494040e+07,3865.281909,2145.250,1.285068,10.839629,160.046038,1401.5,3546.750,6579.20,1206.0,2381.5
3,potrosnja_gb,1178,11.736503,5.910720,4.097366,6.906038,999.5,3.714085e+03,60.943296,7.075,5.190424,15.095391,230.057806,3.3,10.375,19.50,2.3,6.3
4,minuti_poziva,1232,279.222403,NaN,NaN,255.969574,7639.0,1.861534e+05,431.454944,183.250,1.544575,15.225396,253.176346,162.0,345.250,460.45,0.0,258.5
5,broj_sms,1232,38.410714,37.916743,37.417452,38.262677,39.0,3.775969e+01,6.144891,9.000,0.159914,0.225861,-0.129197,34.0,43.000,49.00,37.0,38.0
6,broj_reklamacija,1232,0.758929,0.000000,0.000000,0.572008,7.0,1.068564e+00,1.033713,1.000,1.361517,1.765438,3.991784,0.0,1.000,3.00,0.0,0.0
7,ocena_zadovoljstva,1136,3.617958,3.461693,3.270163,3.665934,4.0,9.640473e-01,0.981859,1.000,0.271265,-0.300381,-0.525673,3.0,4.000,5.00,4.0,4.0


mesecni_racun_rsd [72090.0, 64490.0, 40630.0, 32910.0, 32020.0, 31070.0, 29740.0, 24040.0, 20750.0, 9675.0]
potrosnja_gb [999.9, 999.9, 920.5, 920.5, 850.0, 64.2, 47.9, 39.6, 38.9, 38.8]
minuti_poziva [7500, 7500, 7500, 7500, 728, 700, 671, 612, 610, 610]


/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value encountered in log
  log_a = xp.log(a)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning: The harmonic mean is only defined if all elements are non-negative; otherwise, the result is NaN.
  res = hypotest_fun_out(*samples, **kwds)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value encountered in log
  log_a = xp.log(a)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning: The harmonic mean is only defined if all elements are non-negative; otherwise, the result is NaN.
  res = hypotest_fun_out(*samples, **kwds)
/home/machina/Desktop/praksa-posao/.venv/lib/python3.12/site-packages/scipy/stats/_stats_py.py:232: RuntimeWarning: invalid value enco

# Sumnjive stvari


- Mesečni račun: Imamo vrednost asimetrije raspodele $>10$ što nas navodi da očekujemo da će medijana sigurno da bude manja u odnosu na sredinu uzorka. To smo utvrdili još na početku negde, a ovde se definitivno potvrdilo.

- `broj_reklamacija` u odnosu na `broj_sms` (koef varijacije): Raspodela `broj_poruka` u ovom periodu je kudi kamo više simetrična u odnosu na `broj_reklamacija`.

- Geometrijska sredina: `None` (vidi u pitanjima za mentora)

- Smešne vrednosti za `mesecni_racun_rsd` i `potrosnja_gb`

## Klase problema

U fajlu `problemi.csv` se nalaze sve detektovane klase problema na programski način. Ipak, dodao bih još da:

- `Prepaid` korisnik ne plaća mesečnu pretplatu, a skup podataka to predviđa
- Javljaju se neke oprečne cene za `mesecni_racun_rsd` spram korisnika,

In [60]:
issues=[]

for col in data.columns:
    cond=data[col].isna()
    count = cond.sum()
    if count>0:
        issues.append({
            "kolona":col,
            "klasa":"nedostajuce_vrednosti",
            "broj_problema":count,
            "primeri_problema":data.loc[cond,col].head().tolist()
        })

for col in ["mesecni_racun_rsd","potrosnja_gb"]:
    non_num= pd.to_numeric(data[col],errors="coerce").isna() & data[col].notna()
    issues.append({
            "kolona":col,
            "klasa":"nenumericke_u_numerickim",
            "broj_problema":non_num.sum(),
            "primeri_problema":data.loc[non_num,col].head().tolist()
        })
duplicates=dict()
duplicates["klasa"]="duplikati_cisti"

for col in data.columns:
    cond = data[col].duplicated()
    issues.append({
        "kolona":col,
        "klasa":"duplikati_potpuni",
        "broj_problema":cond.sum(),
        "primeri_problema":data.loc[cond,col].head().unique().tolist()
    })


cat_issues=dict()
cat_issues["klasa"]="nedosledne_kategorije"

correct = {
    "region": [
        "Beograd",
        "Vojvodina",
        "Šumadija i Zapadna Srbija",
        "Južna i Istočna Srbija"
    ],
    "pol": ["M", "Ž"],
    "kanal_prodaje": [
        "Poslovnica",
        "Online",
        "Call centar",
        "Partner"
    ],
    "churn": ["Da", "Ne"],
    "tip_paketa": ["Prepaid", "Postpaid", "Biznis"]
}

for cat in categories:
    cond = data[cat].notna() & ~data[cat].isin(correct[cat])
    if(cond.any()):
        issues.append({
            "kolona": cat,
            "klasa": "nedosledne_kategorije",
            "broj_problema": cond.sum(),
            "primeri_problema": data.loc[cond, cat].value_counts().to_dict()
        })




#for issue in issues:
 #   print(f"{issue}:{issues[issue]}\n")'''

numeric_cols=data.select_dtypes(include="number").columns

for col in numeric_cols:
    cond = data[col]<0
    if cond.any():
        print(data.loc[cond,col].head().tolist())
        issues.append({
            "kolona":col,
            "klasa":"negativne_vrednosti",
            "broj_problema":cond.sum(),
            "primeri_problema":data.loc[cond, col].head().tolist()
        })




[-26.0, -19.0, -21.0]
[-4, -9, -9, -3, -3]
[-139, -98, -62, -14]



- Razlika u duplikatima se javlja zbog `pandas`-ovog pristupa brojanju. Кljučni parametar u ovom slučaju je `keep="first"` (podrazumevano), što znači da tek kad naiđe na prvo sledeće pojavljivanje, onda proglašava stanje imanja duplikata. Dok ako bi stavili `keep=False`, prvo i zadnje pojavljivanje bi se uračunalo što nam je od interesa. [Izvor](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)



In [61]:
date_conv = pd.to_datetime(data.loc[format_ok,"datum_aktivacije"],format="%Y-%m-%d")

date_cond = date_conv > pd.Timestamp("2026-09-01")

if date_cond.any():
    issues.append({
        "kolona":"datum_aktivacije",
        "klasa":"van_opsega",
        "broj_problema":date_cond.sum(),
        "primeri_problema":date_conv.loc[date_cond].head().tolist()
    })



frame = pd.DataFrame(issues)
frame.to_csv("problemi.csv",index=False)